# Select and export VGGT experiments
**Restart the kernel before using this redesigned notebook**, so old model/dataset variables are released. This notebook now only browses TAP-Vid DAVIS and exports selections. It does not import PyTorch/VGGT, load model weights, run inference, or analyse predictions. Your existing `notebook.ipynb` is unchanged.

Workflow: **browse → queue several experiments → export all selections → shut down this kernel → run the terminal script**. An analysis notebook will be added later.

Dataset: `AML_project/tapvid_davis/tapvid_davis.pkl`. Only load a trusted official pickle: pickle can execute code. The whole dictionary must currently be read for browsing, but it is never passed to the runner.

Exports go to `images/videos/<sequence>_<start>_<end>/` (with a stride suffix when needed). The end is exclusive. Existing `.npz` exports are not overwritten. Each `.npz` contains only the selected RGB frames, normalized annotations, original frame/point IDs, and your settings/notes. Model-specific resize/crop is performed later by the runner using VGGT's standard helper, not by the preview. No new datasets or packages are downloaded automatically.


In [1]:
from pathlib import Path
import gc, json, pickle, shlex, sys
from IPython.display import HTML, display

candidates = [Path.cwd(), Path.cwd() / 'AML_project', *Path.cwd().parents]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'steps.md').is_file() and (p / 'README.md').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open the notebook from AML_project, its parent, or a subdirectory.')
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from vggt_inputs import sequence_table, make_video_preview, select_clip, normalize_spec, export_iterations
print('Project:', PROJECT_ROOT)
print('Kernel:', sys.executable)


Project: /home/paul/Documents/Cours/Advanced Machine Learning/AML_project
Kernel: /home/paul/Documents/Cours/Advanced Machine Learning/vggt/venv/bin/python


## 1. Load the trusted benchmark for browsing
Set the trust switch after verifying that your file came from the [official DAVIS archive](https://storage.googleapis.com/dm-tapnet/tapvid_davis.zip). This uses system RAM, not GPU memory. Do not run a model process at the same time.


In [2]:
DATASET_PATH = PROJECT_ROOT / 'tapvid_davis/tapvid_davis.pkl'
TRUST_DATASET_PICKLE = True  # Set True only after verifying your source.
if not DATASET_PATH.is_file():
    raise FileNotFoundError(DATASET_PATH)
if not TRUST_DATASET_PICKLE:
    raise ValueError('Verify the pickle source, then set TRUST_DATASET_PICKLE=True.')
with DATASET_PATH.open('rb') as stream:
    dataset = pickle.load(stream)
dataset_loaded_from = DATASET_PATH.resolve()
display(HTML(sequence_table(dataset)))


Sequence,Frames,Resolution (W x H),Annotated points
bike-packing,69,910 x 480,25
blackswan,50,854 x 480,5
bmx-trees,80,854 x 480,25
breakdance,84,854 x 480,25
camel,90,854 x 480,10
car-roundabout,75,854 x 480,20
car-shadow,40,854 x 480,25
cows,104,854 x 480,10
dance-twirl,90,854 x 480,25
dog,60,854 x 480,10


## 2. Watch a sequence
Choose a name from the table. The player shows every stored frame, independently of the inference frame limit below. Use play/pause and the slider; optionally show visible ground-truth point IDs. These are original annotation IDs, which you can use for `POINT_IDS` below.

Preview FPS is only a viewing speed, not the original recording rate. Preview width affects display only. If JavaScript is blocked by your editor, execute the cell in a trusted JupyterLab session. Clear embedded preview outputs before committing the notebook. No rigid/non-rigid label is assigned automatically.


In [3]:
SEQUENCE_NAME = "paragliding-launch"  # None = first name alphabetically; or choose a name such as 'breakdance'.
PREVIEW_FPS = 10.0
PREVIEW_WIDTH = 640
SHOW_ANNOTATIONS = False


In [4]:
preview_name = sorted(dataset)[0] if SEQUENCE_NAME is None else SEQUENCE_NAME
if preview_name not in dataset:
    raise KeyError(f'Unknown sequence: {preview_name}')
print('Preview:', preview_name, '| stored frames:', len(dataset[preview_name]['video']))
display(HTML(make_video_preview(dataset[preview_name], PREVIEW_FPS, PREVIEW_WIDTH, SHOW_ANNOTATIONS)))


Preview: paragliding-launch | stored frames: 80


## 3. Define and queue experiments
For each desired video/window: select `SEQUENCE_NAME`, inspect its preview, set the options below, then run **Add/update queue**. Repeat for as many experiments as desired before exporting. The queue stores recipes, not copies of all selected videos.

Use a different `EXPERIMENT_NAME` for each recipe in the queue. Reusing a name updates the recipe; the exporter assigns `experiment_NNN.npz` automatically. `START_FRAME` and `FRAME_STRIDE` use the player’s stored frame indices. `MAX_FRAMES=None` selects all remaining sampled frames.

`POINT_IDS=None` exports all annotated points; otherwise supply original IDs such as `[0, 4, 12]`. The runner queries only those visible and inside VGGT's crop in the first selected frame. `MAX_POINTS` optionally caps that eligible set, in the supplied ID order. Ground truth and IDs remain attached to every export. This first-frame-visible protocol is exploratory, not complete official TAP-Vid evaluation.


In [5]:
EXPERIMENT_NAME = 'experiment_001'  # Letters, digits, underscores, hyphens; change for each queued case.
START_FRAME = 11
FRAME_STRIDE = 1
MAX_FRAMES = 20 # None = all remaining sampled frames.
POINT_IDS = None
MAX_POINTS = None
NOTES = ''  # Your own motion classification and reason for selecting this clip.


In [6]:
if 'EXPERIMENTS' not in globals():
    EXPERIMENTS = {}
spec = normalize_spec(dict(name=EXPERIMENT_NAME, sequence=sorted(dataset)[0] if SEQUENCE_NAME is None else SEQUENCE_NAME,
                           start_frame=START_FRAME, frame_stride=FRAME_STRIDE, max_frames=MAX_FRAMES,
                           point_ids=POINT_IDS, max_points=MAX_POINTS, notes=NOTES))
# Validate this selection now, then release its temporary copy.
selected_clip = select_clip(dataset, spec, dataset_loaded_from)
print('Selected stored frame indices:', selected_clip['frame_indices'].tolist())
print('Exported original point IDs:', selected_clip['point_ids'].tolist())
del selected_clip
EXPERIMENTS[spec['name']] = spec
print('Queued experiments:', len(EXPERIMENTS))
print(json.dumps(list(EXPERIMENTS.values()), indent=2))


Selected stored frame indices: [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
Exported original point IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
Queued experiments: 1
[
  {
    "start_frame": 11,
    "frame_stride": 1,
    "max_frames": 20,
    "point_ids": null,
    "max_points": null,
    "notes": "",
    "name": "experiment_001",
    "sequence": "paragliding-launch"
  }
]


Review the printed queue before exporting. To remove a recipe, run `EXPERIMENTS.pop('its_name')` in a cell; to clear the queue, assign `EXPERIMENTS = {}`. These operations affect only the in-memory queue, not exported files.


## 4. Export all queued experiments
Each queued selection goes into the folder for its actual sequence and frame window. The exporter checks the highest `experiment_NNN.npz` number in that folder and adds one (or starts at `001`). It adds each export to the manifest and processes selections one at a time. Rerun this cell to export another iteration without overwriting earlier files.

Only selected raw RGB frames are saved. The runner performs native VGGT preprocessing and maps the selected annotations consistently. It never opens the full DAVIS pickle.


In [7]:
if not globals().get('EXPERIMENTS'):
    raise ValueError('Queue at least one experiment first.')
exported_paths = export_iterations(dataset, list(EXPERIMENTS.values()), dataset_loaded_from,
                                   PROJECT_ROOT / 'images/videos')
# The runner requires unique file stems per command, so group by window folder.
exports_by_window = {}
for path in exported_paths:
    exports_by_window.setdefault(path.parent, []).append(str(path))
print('After shutting down this kernel, run in a TERMINAL (new exports only):')
for files in exports_by_window.values():
    command = [sys.executable, str(PROJECT_ROOT / 'scripts/run_vggt.py'), *files, '--device', 'cuda']
    print(shlex.join(command))
print('To validate inputs without loading VGGT, append --validate-only.')
print('To run on CPU instead, change --device cuda to --device cpu.')


Exported paragliding-launch_11_31/experiment_001.npz: 20 frames, 25 annotated points
After shutting down this kernel, run in a TERMINAL (new exports only):
'/home/paul/Documents/Cours/Advanced Machine Learning/vggt/venv/bin/python' '/home/paul/Documents/Cours/Advanced Machine Learning/AML_project/scripts/run_vggt.py' '/home/paul/Documents/Cours/Advanced Machine Learning/AML_project/images/videos/paragliding-launch_11_31/experiment_001.npz' --device cuda
To validate inputs without loading VGGT, append --validate-only.
To run on CPU instead, change --device cuda to --device cpu.


## 5. Release data, then shut down this kernel
The cell below releases the full dataset. You must reload it if you want to browse/export more selections. The queue contains only small recipes and the exported files remain on disk.

Then explicitly **shut down the kernel** through Jupyter or your editor. Closing the notebook tab alone is insufficient. Run the printed command(s) from a terminal, not `!python ...` inside a still-running notebook. To repeat inference without making another copy of the input, rerun a command on the same `.npz`; the runner creates a fresh results directory each time.


In [8]:
if 'dataset' in globals():
    del dataset
if 'selected_clip' in globals():
    del selected_clip
gc.collect()
print('Dataset released. Shut down this kernel before starting the terminal runner.')


Dataset released. Shut down this kernel before starting the terminal runner.


## What the terminal runner does
- Uses your existing sibling `vggt` source checkout; `--vggt-root` can override its location. No installation occurs.
- Uses the cached `model.pt` checkpoint; `--checkpoint` can select a trusted local checkpoint. Missing weights produce an error, not an automatic download.
- Starts a fresh subprocess for each export. Each child loads only one selected clip and the model, writes results, and exits, releasing memory before the next case.
- Avoids allocating both a randomly initialized full model and a deserialized full checkpoint: constructs parameter shapes on the meta device, memory-maps the checkpoint, and assigns its tensors before device transfer.
- Keeps the default architecture, all heads, FP32 parameters, and native preprocessing. CUDA uses autocasting; CPU uses FP32. No automatic CPU fallback, resolution changes, or frame reduction. CPU can be much slower and still needs RAM.
- Saves arrays and metadata under `results/vggt/<run timestamp>/<experiment name>/`, plus a batch status file. Errors stop the batch by default; `--keep-going` explicitly requests continuing with subsequent experiments. A failed process does not count as a completed result.

Use `python scripts/run_vggt.py --help` for options. Results include model outputs, processed inputs, original point/frame IDs, ground truth mapped to model coordinates, and the original export settings. No analysis notebook or benchmark scoring is included yet.
